In [ ]:
%gui qt
%load_ext autoreload
%autoreload 2

import hmt_v3 as hmt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Preprocessing and Formatting

In [ ]:
plt.rcParams.update({
    # --- text ---
    "font.size": 14,
    # "axes.titlesize": 16,
    # "axes.labelsize": 14,
    # "xtick.labelsize": 12,
    # "ytick.labelsize": 12,
    # "legend.fontsize": 14,
    # "figure.titlesize": 20,

    # --- lines & markers ---
    "lines.linewidth": 2.5,
    "lines.markersize": 8,
    "patch.linewidth": 1.5,      # bar/patch outlines
    "axes.linewidth": 1.5,       # axis spines

    # --- ticks ---
    "xtick.major.width": 1.5,
    "ytick.major.width": 1.5,
    "xtick.major.size": 6,
    "ytick.major.size": 6,

    # --- export quality ---
    "savefig.dpi": 300,
    "figure.dpi": 100,           # on-screen only; doesn't affect saved size
    "savefig.bbox": "tight",
})

## Simulated Data Generation

### Toy centroid model: me3/ac integration sweep

Synthetic centroid generator and spatial-distribution metrics, defined in `hmt_v3/postprocess.py` (independent of the real `me3_df` / `ac_df` and nucleus mask above). `integration_level` is a single knob (0 = fully independent CSR centroids per channel, 1 = every me3 centroid has a paired ac centroid) for testing centroid-based spatial-distribution metrics — colocalization fraction, cross pair-correlation, self-nonself contact ratio — against a known ground truth.

In [ ]:
# --- Generate balanced me3/ac centroid distributions across integration_level ---
rng = np.random.default_rng(0)
field_size_nm = 10000.0
coloc_radius_nm = 150.0
integration_levels = [0.0, 0.25, 0.5, 0.75, 1.0]

sweep = hmt.postprocess.generate_integration_sweep(
    integration_levels, n_domains=150, field_size_nm=field_size_nm, rng=rng)

fig, axes = plt.subplots(1, len(integration_levels), figsize=(4 * len(integration_levels), 4), sharey=True)
for row, ax in zip(sweep, axes):
    coloc = hmt.postprocess.colocalization_fraction(
        row["me3_seeds"][["x [nm]", "y [nm]"]].to_numpy(),
        row["ac_seeds"][["x [nm]", "y [nm]"]].to_numpy(), coloc_radius_nm)
    row["coloc_fraction"] = coloc

    ax.scatter(row["me3_seeds"]["x [nm]"], row["me3_seeds"]["y [nm]"], s=15, alpha=0.7, color="green", label="me3")
    ax.scatter(row["ac_seeds"]["x [nm]"], row["ac_seeds"]["y [nm]"], s=15, alpha=0.7, color="red", label="ac")
    ax.set_title(f"level={row['integration_level']}")
    ax.set_aspect("equal")

axes[0].legend(loc="upper right", fontsize=10)
plt.suptitle("Toy centroids (balanced)")
plt.tight_layout()
plt.show()

The balanced sweep above always gives me3 and ac the same total centroid count, so `me3_expected_ratio`/`ac_expected_ratio` ≈ 1 already and normalizing the self-nonself contact ratio barely changes anything — that's expected, not a bug, since normalization only corrects for a count imbalance that isn't present there. The demo below repeats the sweep with 2x as many me3 centroids as ac (`n_domains=300, n_domains_ac=150`) so the raw curves are dominated by that abundance imbalance while the normalized curves stay comparable to the balanced case.

In [ ]:
# --- Generate imbalanced me3/ac centroid distributions (300 me3 vs. 150 ac) ---
rng = np.random.default_rng(0)
integration_levels = [0.0, 0.25, 0.5, 0.75, 1.0]

imbalanced_sweep = hmt.postprocess.generate_integration_sweep(
    integration_levels, n_domains=300, n_domains_ac=150, field_size_nm=field_size_nm, rng=rng)

fig, axes = plt.subplots(1, len(integration_levels), figsize=(4 * len(integration_levels), 4), sharey=True)
for row, ax in zip(imbalanced_sweep, axes):
    ax.scatter(row["me3_seeds"]["x [nm]"], row["me3_seeds"]["y [nm]"], s=15, alpha=0.7, color="green",
               label=f"me3 (n={len(row['me3_seeds'])})")
    ax.scatter(row["ac_seeds"]["x [nm]"], row["ac_seeds"]["y [nm]"], s=15, alpha=0.7, color="red",
               label=f"ac (n={len(row['ac_seeds'])})")
    ax.set_title(f"level={row['integration_level']}")
    ax.set_aspect("equal")

axes[0].legend(loc="upper right", fontsize=10)
plt.suptitle("Toy centroids (imbalanced, 300 me3 vs. 150 ac)")
plt.tight_layout()
plt.show()

### Self-nonself contact ratio

`self_nonself_contact_ratio` and `plot_sncr_sweep` now live in `hmt_v3/postprocess.py` rather than the notebook. Plotted here on the balanced and imbalanced centroid distributions generated above.

In [ ]:
# --- SNCR: balanced distribution ---
hmt.postprocess.plot_sncr_sweep(
    sweep, field_size_nm, r_max=800.0, dr=25.0,
    title="Self-nonself contact ratio (balanced)")
plt.show()

In [ ]:
# --- SNCR: imbalanced distribution ---
hmt.postprocess.plot_sncr_sweep(
    imbalanced_sweep, field_size_nm, r_max=800.0, dr=25.0,
    title="Self-nonself contact ratio (imbalanced)")
plt.show()

### Ripley's K (bivariate cross-K/L)

`ripleys_k_cross` and `plot_ripley_k_sweep` (`hmt_v3/postprocess.py`) compute the standard bivariate Ripley's K function between me3 and ac centroids -- the disk-cumulative sibling of `cross_pair_correlation` (K is g's running integral), reported via the variance-stabilized `L(r) - r` (0 = CSR, >0 = attraction/integration at scale r, <0 = repulsion). Plotted on the same balanced and imbalanced distributions generated above.

In [ ]:
# --- Ripley's K/L: balanced distribution ---
hmt.postprocess.plot_ripley_k_sweep(
    sweep, field_size_nm, r_max=800.0, dr=25.0,
    title="Ripley's K/L (balanced)")
plt.show()

In [ ]:
# --- Ripley's K/L: imbalanced distribution ---
hmt.postprocess.plot_ripley_k_sweep(
    imbalanced_sweep, field_size_nm, r_max=800.0, dr=25.0,
    title="Ripley's K/L (imbalanced)")
plt.show()

### Graph-theory metrics: Delaunay assortativity & Friedman-Rafsky MST test

Two graph-based views of the same centroids (`hmt_v3/postprocess.py`), neither needing a chosen radius `r`:
- `delaunay_channel_mixing` / `plot_delaunay_sweep`: Delaunay triangulation over the pooled centroids, edges labelled homotypic/heterotypic; reports the raw heterotypic edge fraction and Newman's assortativity coefficient (chance-corrected, 0 = no mixing preference, -1 = maximally mixed, +1 = fully segregated).
- `friedman_rafsky_test` / `plot_mst_sweep`: minimum spanning tree over the pooled centroids, cross-type edges counted and compared to a label-permutation null (a proper two-sample statistical test for whether me3/ac come from the same spatial distribution).

In [ ]:
# --- Delaunay assortativity: balanced distribution ---
hmt.postprocess.plot_delaunay_sweep(sweep, title="Delaunay triangulation (balanced)")
plt.show()

In [ ]:
# --- Delaunay assortativity: imbalanced distribution ---
hmt.postprocess.plot_delaunay_sweep(imbalanced_sweep, title="Delaunay triangulation (imbalanced)")
plt.show()

In [ ]:
# --- Friedman-Rafsky MST test: balanced distribution ---
hmt.postprocess.plot_mst_sweep(sweep, n_permutations=500, rng=np.random.default_rng(0),
                               title="Minimum spanning tree (balanced)")
plt.show()

In [ ]:
# --- Friedman-Rafsky MST test: imbalanced distribution ---
hmt.postprocess.plot_mst_sweep(imbalanced_sweep, n_permutations=500, rng=np.random.default_rng(0),
                               title="Minimum spanning tree (imbalanced)")
plt.show()

### MST merge curve (H0 persistent homology)

A separate view of the same MST used above, from `mst_merge_curve` / `plot_merge_curve_sweep` (`hmt_v3/postprocess.py`). `friedman_rafsky_test` collapses the whole tree into one number (total cross-type edge count vs. a permutation null); this instead replays the MST's edges in increasing length order -- exactly the order components merge under a Vietoris-Rips filtration, i.e. 0-dimensional persistent homology -- and tracks the cumulative fraction of merges that are heterotypic as a function of merge radius. That answers a different question: not "how much cross-channel merging is there overall" but "at what length scale does it start."

In [ ]:
# --- MST merge curve: balanced distribution ---
hmt.postprocess.plot_merge_curve_sweep(sweep, r_max=800.0, title="MST merge curve (balanced)")
plt.show()

In [ ]:
# --- MST merge curve: imbalanced distribution ---
hmt.postprocess.plot_merge_curve_sweep(imbalanced_sweep, r_max=800.0, title="MST merge curve (imbalanced)")
plt.show()

### Separate-mesh overlap

Unlike `delaunay_channel_mixing` above (one triangulation pooling both channels), `mesh_overlap` / `plot_mesh_overlap_sweep` (`hmt_v3/postprocess.py`) build TWO independent triangulations -- one over me3 centroids only, one over ac centroids only -- and ask where the two meshes physically cross in space. This captures whether each channel's local connectivity *structure* threads through the other's, rather than whether individual domains sit next to each other. Includes an optional label-permutation significance test (same logic as Friedman-Rafsky, but rebuilding both triangulations each permutation instead of reshuffling one fixed tree, so it's slower -- kept to 100 permutations here).

In [ ]:
# --- Mesh overlap: balanced distribution ---
hmt.postprocess.plot_mesh_overlap_sweep(sweep, n_permutations=100, rng=np.random.default_rng(0),
                                        title="Delaunay mesh overlap (balanced)")
plt.show()

In [ ]:
# --- Mesh overlap: imbalanced distribution ---
hmt.postprocess.plot_mesh_overlap_sweep(imbalanced_sweep, n_permutations=100, rng=np.random.default_rng(0),
                                        title="Delaunay mesh overlap (imbalanced)")
plt.show()

## Parameter sweeps, density subsampling & perturbation datasets

Everything below is driven by `hmt_v3/sweep.py` (`hmt.sweep`), built on the toy
centroid primitives in `hmt.postprocess`:

* `simulate_pair_centroids` - the symmetric integration model
  (`simulate_toy_centroids`) re-parametrised by a **total** centroid count +
  an **me3/ac balance** instead of two raw per-channel counts.
* `simulate_one_sided_centroids` - **asymmetric** clustering: one channel is a
  clean independent CSR process, the other clusters onto it.
* `summarize_metrics` - runs every centroid metric in `hmt.postprocess` and
  collapses each to scalars, so one (me3, ac) point set -> one DataFrame row
  (column list: `hmt.sweep.METRIC_COLUMNS`).

Four experiments, each writing a tidy CSV to `simulated_data/sweeps/`:

| driver | what moves | what is held fixed |
|---|---|---|
| `oat_parameter_sweep` | one of {field size, coloc radius = pair jitter, integration level, n localizations, me3/ac balance} at a time | the other four, at `hmt.sweep.BASELINE` |
| `one_sided_clustering_sweep` | fraction of the follower channel that clusters onto the independent anchor | everything else, at `BASELINE` |
| `density_subsample_experiment` | sampling density (random subsets of one dense realization) | the underlying point structure |
| `perturbation_experiment` | isotropic Gaussian jitter added to every point | the point set itself |

`summarize_stability(df, group_cols)` reduces any of these to per-group
mean / std / CV.

In [ ]:
# hmt_v3.sweep was added after this notebook was first opened. A plain re-run of
# the import cell at the top will NOT attach a newly added submodule, so force a
# package reload here (restart the kernel instead if anything looks stale).
import importlib
import hmt_v3
importlib.reload(hmt_v3)
import hmt_v3 as hmt

from pathlib import Path
import numpy as np
import pandas as pd

OUT = Path("simulated_data/sweeps")
OUT.mkdir(parents=True, exist_ok=True)

# Analysis-side knobs, held fixed across EVERY experiment below so only the
# generation parameter under test moves. coloc_radius_nm is the measurement
# radius for colocalization_fraction - NOT the generation-time pair jitter,
# which is swept separately as pair_jitter_nm.
ANALYSIS = dict(coloc_radius_nm=150.0, reference_radius_nm=250.0, r_max=800.0, dr=25.0)

print("BASELINE:", hmt.sweep.BASELINE)
for k, v in hmt.sweep.DEFAULT_SWEEP_GRID.items():
    print(f"  {k:18s} {v}")

### 1. One-at-a-time parameter sweep

Each of the five basic parameters is swept across
`hmt.sweep.DEFAULT_SWEEP_GRID` while the other four stay at
`hmt.sweep.BASELINE`; `n_replicates` independent realizations per grid point,
all from one seed. The r-based metrics use a fixed absolute `r_max` /
`reference_radius_nm` across every panel so their scalars stay comparable - so
in the `field_size_nm` panel the smallest fields keep a smaller border-safe
fraction of their points (a mild confound of that panel only).

In [ ]:
# ~250 datasets x (PCF / SNCR / Ripley / Delaunay / MST / mesh + 200-perm
# Friedman-Rafsky). Expect a few minutes. Drop fr_permutations / n_replicates
# for a faster first pass.
oat = hmt.sweep.oat_parameter_sweep(
    n_replicates=10, seed=0, fr_permutations=200, mesh_permutations=0, **ANALYSIS)
oat.to_csv(OUT / "oat_parameter_sweep.csv", index=False)
print(oat.shape)
oat.groupby(["swept_param", "swept_value"])[
    ["coloc_frac_me3", "gcross_peak", "delaunay_assortativity", "fr_z"]].mean()

### 2. One-sided (asymmetric) clustering

The anchor channel is a clean independent CSR process; the follower channel
places `cluster_fraction` of its points onto randomly chosen anchor points.
Run for `follower="me3"` (ac is the clean anchor) and `follower="ac"`. Compare
against `oat[oat.swept_param == "integration_level"]`, which is the symmetric
version at the same nominal levels - metrics that only see "are the two
channels near each other" will look similar; metrics that see *which* channel
carries the clustering structure will not.

In [ ]:
one_sided = hmt.sweep.one_sided_clustering_sweep(
    cluster_fractions=(0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    followers=("me3", "ac"), n_replicates=10, seed=0,
    fr_permutations=200, mesh_permutations=0, **ANALYSIS)
one_sided.to_csv(OUT / "one_sided_clustering_sweep.csv", index=False)
print(one_sided.shape)
one_sided.groupby(["follower", "cluster_fraction"])[
    ["coloc_frac_me3", "coloc_frac_ac", "sncr_me3_norm_at_ref", "sncr_ac_norm_at_ref",
     "delaunay_assortativity", "fr_z"]].mean()

### 3. Density robustness (subsampling one fixed structure)

Generate one dense realization, then draw random subsets at a range of
retention fractions (`n_draws` each). Because every draw at a given fraction is
a subset of the *same* points, the spread across draws is pure
sampling/estimator noise and any drift of the per-fraction mean is genuine
density-dependence of the metric (e.g. Friedman-Rafsky `fr_z` scales with n and
is *not* density-stable; `delaunay_assortativity` largely is).

In [ ]:
rng = np.random.default_rng(0)
me3_dense, ac_dense = hmt.sweep.simulate_pair_centroids(
    n_total=6 * hmt.sweep.BASELINE["n_total"], me3_fraction=0.5, integration_level=0.5,
    field_size_nm=hmt.sweep.BASELINE["field_size_nm"],
    pair_jitter_nm=hmt.sweep.BASELINE["pair_jitter_nm"], rng=rng)
me3_dense.to_csv(OUT / "density_dense_me3.csv", index=False)
ac_dense.to_csv(OUT / "density_dense_ac.csv", index=False)

density = hmt.sweep.density_subsample_experiment(
    me3_dense, ac_dense, field_size_nm=hmt.sweep.BASELINE["field_size_nm"],
    fractions=(1.0, 0.75, 0.5, 0.35, 0.2, 0.1, 0.05), n_draws=15, seed=1,
    fr_permutations=100, mesh_permutations=0, **ANALYSIS)
density.to_csv(OUT / "density_subsample.csv", index=False)

density_stability = hmt.sweep.summarize_stability(density, ["fraction"])
density_stability.to_csv(OUT / "density_stability.csv", index=False)
print(density.shape)
density_stability[["fraction", "n_me3_mean",
                   "coloc_frac_me3_mean", "coloc_frac_me3_std",
                   "delaunay_assortativity_mean", "delaunay_assortativity_std",
                   "fr_z_mean", "gcross_peak_mean", "gcross_peak_std"]]

### 4. Positional perturbation

One baseline dataset, then `n_repeats` copies at each `sigma_nm` with every
point independently displaced by an isotropic 2D Gaussian of that stdev
(clipped back into the field). `sigma_nm == 0` is the untouched reference.
`summarize_stability` -> mean / std / CV per sigma ranks the metrics by how
little they move under localization-scale noise.

In [ ]:
rng = np.random.default_rng(0)
me3_base, ac_base = hmt.sweep.simulate_pair_centroids(
    n_total=hmt.sweep.BASELINE["n_total"], me3_fraction=0.5, integration_level=0.5,
    field_size_nm=hmt.sweep.BASELINE["field_size_nm"],
    pair_jitter_nm=hmt.sweep.BASELINE["pair_jitter_nm"], rng=rng)
me3_base.to_csv(OUT / "perturbation_base_me3.csv", index=False)
ac_base.to_csv(OUT / "perturbation_base_ac.csv", index=False)

perturbation = hmt.sweep.perturbation_experiment(
    me3_base, ac_base, field_size_nm=hmt.sweep.BASELINE["field_size_nm"],
    sigmas_nm=(2.5, 5, 10, 20, 40, 80, 160), n_repeats=40, seed=1,
    fr_permutations=100, mesh_permutations=0, **ANALYSIS)
perturbation.to_csv(OUT / "perturbation.csv", index=False)

perturbation_stability = hmt.sweep.summarize_stability(perturbation, ["sigma_nm"])
perturbation_stability.to_csv(OUT / "perturbation_stability.csv", index=False)
print(perturbation.shape)
_cv = [c for c in perturbation_stability.columns if c.endswith("_cv")]
perturbation_stability[["sigma_nm"] + _cv].round(3)

### Overview: metric drift under subsampling vs. positional noise

Each metric normalized to its full-data (left) / zero-noise (right) value, so a
flat line = robust. Metrics whose reference value is ~0 are skipped.

In [ ]:
import matplotlib.pyplot as plt

plot_metrics = ["coloc_frac_me3", "gcross_peak", "gcross_auc",
                "sncr_me3_norm_at_ref", "ripley_Lab_max", "delaunay_assortativity",
                "fr_z", "mst_heterotypic_minus_chance", "mesh_me3_crossing_fraction"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

d = density.groupby("fraction")[plot_metrics].mean()
ref = d.loc[d.index.max()]
for m in plot_metrics:
    if not np.isfinite(ref[m]) or abs(ref[m]) < 1e-9:
        continue
    axes[0].plot(d.index, d[m] / ref[m], marker="o", label=m)
axes[0].axhline(1.0, color="gray", ls="--")
axes[0].invert_xaxis()
axes[0].set_xlabel("retained fraction of localizations")
axes[0].set_ylabel("metric / full-data value")
axes[0].set_title("density subsampling (same structure)")

p = perturbation.groupby("sigma_nm")[plot_metrics].mean()
ref = p.loc[0.0]
for m in plot_metrics:
    if not np.isfinite(ref[m]) or abs(ref[m]) < 1e-9:
        continue
    axes[1].plot(p.index, p[m] / ref[m], marker="o", label=m)
axes[1].axhline(1.0, color="gray", ls="--")
axes[1].set_xlabel("positional noise sigma (nm)")
axes[1].set_title("positional perturbation (same points)")
axes[1].legend(fontsize=8, loc="center left", bbox_to_anchor=(1.02, 0.5))

plt.tight_layout()
plt.show()

## Per-sweep metric visualizations

One multi-panel figure per experiment, each panel a single metric (nine picked
from `METRIC_COLUMNS`), line = mean over replicates, shaded band = ±1 std.
Cells read the tidy CSVs from `simulated_data/sweeps/`, so they run without
re-executing the slow sweeps above.

1. **OAT parameter sweep** — five figures, one per swept parameter, metric vs.
   `swept_value` (other four params at `BASELINE`).
2. **One-sided clustering** — metric vs. `cluster_fraction`, one line per
   `follower` channel (which channel clusters onto the clean CSR anchor).
3. **Density subsampling** — metric vs. retained fraction (x inverted), same
   underlying points; drift = genuine density-dependence.
4. **Positional perturbation** — metric vs. jitter σ (log x), dashed line = the
   untouched σ=0 value.
5. **Robustness heatmap** — within-experiment CV per metric for the density and
   perturbation runs (lower = more stable).

In [ ]:
# === Per-sweep visualizations: load the tidy CSVs and set up a shared panel grid ===
# Self-contained: reads the CSVs written by the four driver cells above, so this
# block can run without re-executing the (slow) sweeps.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

OUT = Path("simulated_data/sweeps")
oat          = pd.read_csv(OUT / "oat_parameter_sweep.csv")
one_sided    = pd.read_csv(OUT / "one_sided_clustering_sweep.csv")
density      = pd.read_csv(OUT / "density_subsample.csv")
density_stab = pd.read_csv(OUT / "density_stability.csv")
perturbation = pd.read_csv(OUT / "perturbation.csv")
perturb_stab = pd.read_csv(OUT / "perturbation_stability.csv")

# metric column -> short label; this dict's order defines every panel grid below.
METRIC_LABELS = {
    "coloc_frac_me3":               "coloc frac (me3->ac)",
    "coloc_frac_ac":                "coloc frac (ac->me3)",
    "gcross_peak":                  "g_cross peak",
    "gcross_auc":                   "g_cross AUC (nm)",
    "sncr_me3_norm_at_ref":         "SNCR me3 norm @ref",
    "ripley_Lab_max":               "Ripley L_ab max",
    "delaunay_assortativity":       "Delaunay assortativity",
    "fr_z":                         "Friedman-Rafsky z",
    "mst_heterotypic_minus_chance": "MST hetero - chance",
}
METRICS = list(METRIC_LABELS)


def panel_grid(df, x, metrics=METRICS, group=None, ncols=3, figsize=None,
               logx=False, invertx=False, ref=None, suptitle=None):
    """One panel per metric. Line = mean over replicates at each x; shaded band
    = +/-1 std. `group` draws one colored line per level of that column.
    `ref` = {metric: value} draws a dashed grey baseline (e.g. the sigma=0 row)."""
    nrows = int(np.ceil(len(metrics) / ncols))
    fig, axes = plt.subplots(nrows, ncols, squeeze=False,
                             figsize=figsize or (5.2 * ncols, 3.5 * nrows))
    levels = [None] if group is None else sorted(df[group].dropna().unique())
    for i, m in enumerate(metrics):
        ax = axes[i // ncols][i % ncols]
        for lev in levels:
            sub = df if lev is None else df[df[group] == lev]
            g = sub.groupby(x)[m].agg(["mean", "std"]).sort_index()
            line, = ax.plot(g.index, g["mean"], marker="o",
                            label=None if lev is None else f"{group}={lev}")
            ax.fill_between(g.index, g["mean"] - g["std"], g["mean"] + g["std"],
                            alpha=0.2, color=line.get_color())
        if ref is not None and m in ref and np.isfinite(ref[m]):
            ax.axhline(ref[m], color="gray", ls="--", lw=1.5)
        if logx:
            ax.set_xscale("log")
        if invertx:
            ax.invert_xaxis()
        ax.set_title(METRIC_LABELS[m], fontsize=12)
        ax.set_xlabel(x)
    for j in range(len(metrics), nrows * ncols):
        axes[j // ncols][j % ncols].set_visible(False)
    if levels != [None]:
        axes[0][0].legend(fontsize=9)
    if suptitle:
        fig.suptitle(suptitle, fontsize=14, y=1.003)
    fig.tight_layout()
    return fig, axes


In [ ]:
# === 1. One-at-a-time parameter sweep: every metric vs. each swept parameter ===
# One figure per swept parameter; line = mean over replicates, band = +/-1 std.
OAT_PARAMS = ["field_size_nm", "pair_jitter_nm", "integration_level", "n_total", "me3_fraction"]
for pname in OAT_PARAMS:
    sub = oat[oat.swept_param == pname]
    panel_grid(sub, x="swept_value",
               suptitle=f"OAT sweep -- {pname}  (other 4 params held at BASELINE; "
                        f"{sub.replicate.nunique()} reps/point)")
    plt.show()


In [ ]:
# === 2. One-sided (asymmetric) clustering: me3-follower vs. ac-follower ===
# Two lines per panel: which channel does the clustering onto the clean anchor.
panel_grid(one_sided, x="cluster_fraction", group="follower",
           suptitle="One-sided clustering  (follower clusters onto clean CSR anchor; band = +/-1 std)")
plt.show()


In [ ]:
# === 3. Density subsampling: metric drift as one fixed structure is thinned ===
# x inverted so the series reads left->right as "keep fewer points".
panel_grid(density, x="fraction", invertx=True,
           suptitle="Density subsampling  (same points, band = +/-1 std over draws; x: 1.0 -> 0.05)")
plt.show()


In [ ]:
# === 4. Positional perturbation: metric drift under Gaussian localization noise ===
# x = jitter stdev (log scale); dashed grey line = the untouched sigma=0 value.
ref0 = perturbation.loc[perturbation.sigma_nm == 0, METRICS].iloc[0].to_dict()
panel_grid(perturbation[perturbation.sigma_nm > 0], x="sigma_nm", logx=True, ref=ref0,
           suptitle="Positional perturbation  (dashed = sigma=0 reference, band = +/-1 std over repeats)")
plt.show()


In [ ]:
# === 5. Robustness overview: coefficient of variation (std / |mean|) per metric ===
# Reads the *_stability.csv files (already reduced to per-group mean/std/CV).
# Lower CV = the metric barely moves within that group -> more robust.
cv_cols = [f"{m}_cv" for m in METRICS]


def _cv_frame(stab, key):
    cols = [c for c in cv_cols if c in stab.columns]
    f = stab.set_index(key)[cols]
    f.columns = [METRIC_LABELS[c[:-3]] for c in cols]
    return f.T  # metrics on rows, group levels on columns


dcv = _cv_frame(density_stab, "fraction")
pcv = _cv_frame(perturb_stab, "sigma_nm")
vmax = np.nanpercentile(np.concatenate([dcv.values.ravel(), pcv.values.ravel()]), 90)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, mat, ttl in [(axes[0], dcv, "density subsampling (by retained fraction)"),
                     (axes[1], pcv, "positional perturbation (by sigma_nm)")]:
    im = ax.imshow(mat.values, aspect="auto", cmap="magma_r", vmin=0, vmax=vmax)
    ax.set_xticks(range(mat.shape[1]))
    ax.set_xticklabels([f"{c:g}" for c in mat.columns])
    ax.set_yticks(range(mat.shape[0]))
    ax.set_yticklabels(mat.index)
    ax.set_title(ttl, fontsize=12)
    for (r, c), v in np.ndenumerate(mat.values):
        if np.isfinite(v):
            ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if v > vmax / 2 else "black")
    fig.colorbar(im, ax=ax, label="CV")
fig.suptitle("Metric stability within each experiment  (lower = more robust)", fontsize=14)
fig.tight_layout()
plt.show()
